In [1]:
#============================================
# Celda 1 — Carga parquet raw
#============================================
import pandas as pd, numpy as np, os
from pathlib import Path

ROOT = Path(os.getcwd()).parent.parent
os.chdir(ROOT)
print(f"✅ ROOT: {ROOT}")

PATH_RAW = ROOT / 'output/idealista/01-raw/raw_idealista_anuncios.parquet'

if not PATH_RAW.exists():
    raise FileNotFoundError(f"❌ No encontrado: {PATH_RAW}\n   Ejecuta primero 01_idealista.ipynb")

df = pd.read_parquet(PATH_RAW)
print(f"✅ Cargado: {df.shape[0]} anuncios × {df.shape[1]} columnas")
print(f"   Cities : {sorted(df['city'].unique())}")
print(f"   Ops    : {sorted(df['operation'].unique())}")
print(f"   Nulos %: {round(df.isnull().sum().sum()/df.size*100,1)}%")
df.head()

✅ ROOT: /workspaces/TFG_Spain-s_Migratory_Flow
✅ Cargado: 700 anuncios × 20 columnas
   Cities : ['barcelona', 'bilbao', 'madrid', 'malaga', 'sevilla', 'valencia', 'zaragoza']
   Ops    : ['rent', 'sale']
   Nulos %: 2.0%


,snapshot_date,city,operation,propertyCode,price,size_m2,price_m2,rooms,bathrooms,floor,exterior,district,municipality,neighborhood,province,latitude,longitude,newDevelopment,status,url
0,2026-06-20,madrid,sale,107795847,1160000.0,218.0,5321.10,3,3,4,True,Hortaleza,Madrid,Virgen del Cortijo - Manoteras,Madrid,40.483829,-3.668391,False,good,https://www.idealista.com/inmueble/107795847/
1,2026-06-20,madrid,sale,110744356,595000.0,48.0,12395.83,2,1,1,False,Barrio de Salamanca,Madrid,Lista,Madrid,40.430963,-3.671766,False,good,https://www.idealista.com/inmueble/110744356/
2,2026-06-20,madrid,sale,109947740,2100000.0,137.0,15328.47,3,3,3,True,Barrio de Salamanca,Madrid,Goya,Madrid,40.422605,-3.677340,False,good,https://www.idealista.com/inmueble/109947740/
3,2026-06-20,madrid,sale,104939115,1980000.0,165.0,12000.00,2,3,7,False,Barrio de Salamanca,Madrid,Goya,Madrid,40.421573,-3.678723,False,good,https://www.idealista.com/inmueble/104939115/
4,2026-06-20,madrid,sale,110955919,3895000.0,205.0,19000.00,3,4,1,True,Barrio de Salamanca,Madrid,Recoletos,Madrid,40.427357,-3.686354,False,good,https://www.idealista.com/inmueble/110955919/


In [2]:
#============================================
# Celda 2 — Limpieza general
#============================================

# Tipos correctos
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'], errors='coerce')

cols_num = ['price','size_m2','price_m2','rooms','bathrooms','latitude','longitude']
for c in cols_num:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

cols_bool = ['exterior','newDevelopment']
for c in cols_bool:
    if c in df.columns:
        df[c] = df[c].astype('boolean')

# Deduplicar por código de anuncio
antes = len(df)
df = df.drop_duplicates(subset=['propertyCode'])
print(f"   Duplicados eliminados: {antes - len(df)}")

# Limpiar strings
for c in ['city','operation','district','municipality','neighborhood','province','status']:
    if c in df.columns:
        df[c] = df[c].str.strip().str.lower()

print(f"✅ df limpio: {df.shape}")
print(f"   Nulos %: {round(df.isnull().sum().sum()/df.size*100,1)}%")
df.head()

   Duplicados eliminados: 0
✅ df limpio: (700, 20)
   Nulos %: 2.0%


,snapshot_date,city,operation,propertyCode,price,size_m2,price_m2,rooms,bathrooms,floor,exterior,district,municipality,neighborhood,province,latitude,longitude,newDevelopment,status,url
0,2026-06-20,madrid,sale,107795847,1160000.0,218.0,5321.10,3,3,4,True,hortaleza,madrid,virgen del cortijo - manoteras,madrid,40.483829,-3.668391,False,good,https://www.idealista.com/inmueble/107795847/
1,2026-06-20,madrid,sale,110744356,595000.0,48.0,12395.83,2,1,1,False,barrio de salamanca,madrid,lista,madrid,40.430963,-3.671766,False,good,https://www.idealista.com/inmueble/110744356/
2,2026-06-20,madrid,sale,109947740,2100000.0,137.0,15328.47,3,3,3,True,barrio de salamanca,madrid,goya,madrid,40.422605,-3.677340,False,good,https://www.idealista.com/inmueble/109947740/
3,2026-06-20,madrid,sale,104939115,1980000.0,165.0,12000.00,2,3,7,False,barrio de salamanca,madrid,goya,madrid,40.421573,-3.678723,False,good,https://www.idealista.com/inmueble/104939115/
4,2026-06-20,madrid,sale,110955919,3895000.0,205.0,19000.00,3,4,1,True,barrio de salamanca,madrid,recoletos,madrid,40.427357,-3.686354,False,good,https://www.idealista.com/inmueble/110955919/


In [3]:
#============================================
# Celda 3 — Imputación de nulos
#============================================

# floor: convertir TODO a string limpio para evitar tipo mixto
# (la columna tiene 'bj', 'ático', '1', '2'... y nulos)
df['floor'] = df['floor'].fillna('desconocido').astype(str).str.strip()

# exterior: nulo → False, luego bool puro
df['exterior'] = df['exterior'].fillna(False).astype(bool)

# district/neighborhood: nulo → 'desconocido'
df['district']     = df['district'].fillna('desconocido').astype(str)
df['neighborhood'] = df['neighborhood'].fillna('desconocido').astype(str)

# Outliers price_m2 — eliminar extremos (<1% y >99%)
for op in df['operation'].unique():
    mask = df['operation'] == op
    q01  = df.loc[mask, 'price_m2'].quantile(0.01)
    q99  = df.loc[mask, 'price_m2'].quantile(0.99)
    antes = mask.sum()
    df = df[~(mask & ((df['price_m2'] < q01) | (df['price_m2'] > q99)))]
    print(f"   [{op}] outliers eliminados: {antes - (df['operation']==op).sum()}")

# Verificar que no quedan tipos mixtos
tipos_problematicos = df.select_dtypes(include='object').apply(
    lambda col: col.dropna().map(type).nunique() > 1
)
if tipos_problematicos.any():
    print(f"⚠️  Columnas con tipos mixtos: {list(tipos_problematicos[tipos_problematicos].index)}")
else:
    print("✅ Sin tipos mixtos — parquet serializable")

print(f"\n✅ Tras imputación y outliers: {df.shape}")
print(f"   Nulos %: {round(df.isnull().sum().sum()/df.size*100,1)}%")

   [sale] outliers eliminados: 7
   [rent] outliers eliminados: 7
✅ Sin tipos mixtos — parquet serializable

✅ Tras imputación y outliers: (686, 20)
   Nulos %: 0.0%


/tmp/ipykernel_28781/2520347354.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  tipos_problematicos = df.select_dtypes(include='object').apply(


In [4]:
#============================================
# Celda 4 — Features derivadas
#============================================

# Ratio precio/habitación
df['price_per_room'] = (df['price'] / df['rooms'].replace(0, np.nan)).round(2)

# Categoría de tamaño
bins_size   = [0, 50, 80, 120, 200, 9999]
labels_size = ['micro','pequeño','mediano','grande','lujo']
df['size_cat'] = pd.cut(df['size_m2'], bins=bins_size, labels=labels_size)

# Categoría de precio/m2 por operación
for op, bins, labels in [
    ('sale',  [0,2000,4000,6000,9000,99999], ['muy_bajo','bajo','medio','alto','premium']),
    ('rent',  [0,8,13,18,25,999],            ['muy_bajo','bajo','medio','alto','premium']),
]:
    mask = df['operation'] == op
    df.loc[mask, 'precio_cat'] = pd.cut(
        df.loc[mask, 'price_m2'], bins=bins, labels=labels
    )

print(f"✅ Features añadidas: price_per_room, size_cat, precio_cat")
df[['city','operation','price','size_m2','price_m2','size_cat','precio_cat']].head(10)

✅ Features añadidas: price_per_room, size_cat, precio_cat


,city,operation,price,size_m2,price_m2,size_cat,precio_cat
0,madrid,sale,1160000.0,218.0,5321.10,lujo,medio
1,madrid,sale,595000.0,48.0,12395.83,micro,premium
2,madrid,sale,2100000.0,137.0,15328.47,grande,premium
3,madrid,sale,1980000.0,165.0,12000.00,grande,premium
5,madrid,sale,1890000.0,194.0,9742.27,grande,premium
6,madrid,sale,850000.0,96.0,8854.17,mediano,alto
7,madrid,sale,995000.0,117.0,8504.27,mediano,alto
8,madrid,sale,1280000.0,247.0,5182.19,lujo,medio
9,madrid,sale,695000.0,80.0,8687.50,pequeño,alto
10,madrid,sale,900000.0,155.0,5806.45,grande,medio


In [5]:
#============================================
# Celda 5 — Resumen calidad final
#============================================
resumen = pd.DataFrame([{
    'tabla'    : 'idealista_anuncios',
    'filas'    : len(df),
    'columnas' : len(df.columns),
    'cities'   : df['city'].nunique(),
    'ops'      : df['operation'].nunique(),
    'snapshot' : df['snapshot_date'].max().date(),
    'nulos_%'  : round(df.isnull().sum().sum() / df.size * 100, 1)
}])
print(resumen.to_string(index=False))

print(f"\nPrice/m2 por operación:")
print(df.groupby('operation')['price_m2'].agg(['mean','median','min','max']).round(2))

             tabla  filas  columnas  cities  ops   snapshot  nulos_%
idealista_anuncios    686        23       7    2 2026-06-20      0.1

Price/m2 por operación:
              mean   median     min       max
operation                                    
rent         20.41    17.78     7.5     58.35
sale       5900.81  5470.59  1388.1  15328.47


In [6]:
#============================================
# Celda 6 — Guardar parquet clean
#============================================
import pyarrow as pa

# ── Sanitizar columnas object con tipos mixtos ──
for col in df.select_dtypes(include=['object', 'str']).columns:
    tipos = df[col].dropna().map(type).unique()
    if len(tipos) > 1:
        print(f"   ⚠️  Tipo mixto en '{col}': {tipos} → convirtiendo a str")
        df[col] = df[col].astype(str)

# ── Sanitizar columnas pd.Categorical (pueden tener problemas con pyarrow) ──
for col in df.select_dtypes(include='category').columns:
    df[col] = df[col].astype(str)

os.makedirs(str(ROOT / 'output/idealista/02-clean'), exist_ok=True)
ruta_clean = str(ROOT / 'output/idealista/02-clean/clean_idealista_anuncios.parquet')
df.to_parquet(ruta_clean, index=False)

print(f"✅ Guardado: output/idealista/02-clean/clean_idealista_anuncios.parquet")
print(f"   Filas   : {len(df)}")
print(f"   Columnas: {len(df.columns)}")
print(f"\n🎯 Pipeline 02_idealista completado — listo para análisis")

✅ Guardado: output/idealista/02-clean/clean_idealista_anuncios.parquet


   Filas   : 686
   Columnas: 23

🎯 Pipeline 02_idealista completado — listo para análisis
